# Discovering quadratic reciprocity from prime statistics

For a fixed integer $a$ and an odd prime $p$, the Legendre symbol

$$X_a(p)=\left(\frac ap\right)$$

records whether $a$ is a square modulo $p$. We will treat these symbols as data: first look at their apparent randomness, then ask how much knowing $p\bmod m$ helps us predict them. The aim is to **discover** a law, not to assume one.

We deliberately compute every symbol with Euler's criterion. Quadratic reciprocity will appear only after the experiments suggest it.

## 1. Setup and data

The loader accepts the specification's `primes.npy` name and the large prime file already included beside this notebook. Memory mapping avoids loading the entire large array.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import gcd, log
from pathlib import Path
from collections import Counter

plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True})

N = 100_000
A_VALUES = [2, 3, 5, 7]
MAX_MODULUS = 60

data_candidates = [Path("primes.npy"), Path("primes_up_to_1000000000.npy")]
prime_path = next((path for path in data_candidates if path.exists()), None)
if prime_path is None:
    raise FileNotFoundError(f"Put a prime array at one of: {data_candidates}")

primes_all = np.load(prime_path, mmap_mode="r")
primes = np.asarray(primes_all[:N], dtype=np.int64)

print("Data file:", prime_path)
print("Number of primes:", len(primes))
print("Smallest prime:", primes[0])
print("Largest prime:", primes[-1])

Data file: primes_up_to_1000000000.npy
Number of primes: 100000
Smallest prime: 2
Largest prime: 1299709


## 2. Euler's criterion

For an odd prime $p$,

$$a^{(p-1)/2}\equiv \left(\frac ap\right)\pmod p.$$

Thus the modular power is $1$ for a residue and $p-1$ for a nonresidue. This is independent of quadratic reciprocity.

In [2]:
def legendre_symbol(a, p):
    """Return the Legendre symbol (a/p), using Euler's criterion."""
    a, p = int(a), int(p)
    if a % p == 0:
        return 0
    r = pow(a, (p - 1) // 2, p)
    if r == 1:
        return 1
    if r == p - 1:
        return -1
    raise ValueError("Unexpected value in Euler criterion")

checks = [(2, 3), (2, 7), (3, 11), (5, 5), (5, 11)]
pd.DataFrame([(a, p, legendre_symbol(a, p)) for a, p in checks],
             columns=["a", "p", "(a/p)"])

,a,p,(a/p)
0,2,3,-1
1,2,7,1
2,3,11,1
3,5,5,0
4,5,11,1


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


> Before continuing, predict a few entries by listing the nonzero squares modulo a small prime. Do the two methods agree?

## 3. Experiment 1 — do the signs look random?

We compute each symbol once and cache it. Primes dividing $2a$ are excluded from binary experiments, since their symbol is $0$ (and $p=2$ is outside the Legendre-symbol setting).

In [ ]:
def valid_prime_symbol_arrays(a, prime_array=primes):
    ps = np.asarray([p for p in prime_array if p != 2 and a % int(p) != 0], dtype=np.int64)
    xs = np.fromiter((legendre_symbol(a, p) for p in ps), dtype=np.int8, count=len(ps))
    return ps, xs

symbol_cache = {a: valid_prime_symbol_arrays(a) for a in A_VALUES}

records = []
for a, (ps, xs) in symbol_cache.items():
    records.extend(zip(ps, [a] * len(ps), xs))
legendre_data = pd.DataFrame(records, columns=["p", "a", "legendre"])

summary_rows = []
for a, (_, xs) in symbol_cache.items():
    plus, minus = int(np.sum(xs == 1)), int(np.sum(xs == -1))
    summary_rows.append((a, plus, minus, plus / len(xs), minus / len(xs)))
summary = pd.DataFrame(summary_rows, columns=["a", "+1 count", "-1 count", "fraction +1", "fraction -1"])
summary.style.format({"fraction +1": "{:.4f}", "fraction -1": "{:.4f}"})

> The two signs occur with nearly equal frequency. Does that mean quadratic residuosity is random, or have we discarded information that makes it predictable?

## 4. Experiment 2 — convergence toward $1/2$

The running fraction is noisy early on. A logarithmic horizontal axis lets us see both the early fluctuations and the long-run behavior.

In [ ]:
fig, ax = plt.subplots()
for a, (_, xs) in symbol_cache.items():
    running = np.cumsum(xs == 1) / np.arange(1, len(xs) + 1)
    ax.plot(np.arange(1, len(xs) + 1), running, label=f"a={a}", linewidth=1)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="1/2")
ax.set(xscale="log", xlabel="number of valid primes", ylabel="running fraction with (a/p)=1",
       title="Apparently fair signs")
ax.legend()
plt.show()

> How quickly does each curve approach $1/2$? Does global balance rule out a deterministic pattern inside congruence classes?

## 5. Experiment 3 — inspect the raw sequence

Start with the first 40 values of $(2/p)$. Then expose one extra feature: $p\bmod 8$.

In [ ]:
p2, x2 = symbol_cache[2]
raw_2 = pd.DataFrame({"p": p2[:40], "(2/p)": x2[:40]})
display(raw_2)

with_mod_8 = raw_2.copy()
with_mod_8.insert(1, "p mod 8", with_mod_8["p"] % 8)
display(with_mod_8)

> Is there an obvious rule in the first table? What changes in the second?

The sequence that looked irregular becomes deterministic after conditioning on $pmod 8$. We now turn that observation into a general search.

## 6. Experiment 4 — conditional distributions modulo $m$

The function below excludes primes dividing $am$. This removes exceptional classes caused by the prime factors of the parameters.

In [ ]:
def symbols_for(a, prime_array):
    if prime_array is primes and a in symbol_cache:
        return symbol_cache[a]
    return valid_prime_symbol_arrays(a, prime_array)

def conditional_legendre_table(a, prime_array, m):
    ps, xs = symbols_for(a, prime_array)
    keep = np.array([gcd(int(p), a * m) == 1 for p in ps])
    residues, values = ps[keep] % m, xs[keep]
    frame = pd.DataFrame({"residue": residues, "legendre": values})
    result = frame.groupby("residue")["legendre"].agg(
        count="size",
        plus_count=lambda s: int((s == 1).sum()),
        minus_count=lambda s: int((s == -1).sum()),
    ).reset_index()
    result["fraction_plus"] = result["plus_count"] / result["count"]
    result["fraction_minus"] = result["minus_count"] / result["count"]
    return result

conditional_legendre_table(2, primes, 8)

## 7. Experiments 5–10 — discover the right modulus

For every residue class, a majority-vote predictor guesses its most common sign. Its in-sample accuracy is

$$A_a(m)=\frac{\text{correct predictions}}{\text{observations}}.$$

Conditional entropy measures the remaining uncertainty:

$$H(X_a\mid R_m)=\sum_r P(R_m=r)h(P(X_a=1\mid R_m=r)).$$

Values near 1 bit mean little information; 0 bits means perfect prediction.

In [ ]:
def prediction_accuracy(a, prime_array, m):
    table = conditional_legendre_table(a, prime_array, m)
    return table[["plus_count", "minus_count"]].max(axis=1).sum() / table["count"].sum()

def binary_entropy(q):
    if q <= 0 or q >= 1:
        return 0.0
    return -q * np.log2(q) - (1 - q) * np.log2(1 - q)

def conditional_entropy(a, prime_array, m):
    table = conditional_legendre_table(a, prime_array, m)
    weights = table["count"] / table["count"].sum()
    return float(np.sum(weights * table["fraction_plus"].map(binary_entropy)))

def find_smallest_predictive_modulus(a, prime_array, max_modulus=100, tolerance=1e-12):
    for m in range(2, max_modulus + 1):
        if conditional_entropy(a, prime_array, m) < tolerance:
            return m
    return None

moduli = np.arange(2, MAX_MODULUS + 1)
search_rows = []
for a in A_VALUES:
    for m in moduli:
        search_rows.append((a, m, prediction_accuracy(a, primes, m),
                            conditional_entropy(a, primes, m)))
search = pd.DataFrame(search_rows, columns=["a", "modulus", "accuracy", "entropy"])

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, a in zip(axes.flat, A_VALUES):
    part = search[search.a == a]
    ax.plot(part.modulus, part.accuracy, marker=".")
    ax.axhline(1, color="black", linestyle="--", linewidth=1)
    ax.set(title=f"a={a}", xlabel="modulus m", ylabel="prediction accuracy")
fig.suptitle("How predictable is (a/p) from p mod m?")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, a in zip(axes.flat, A_VALUES):
    part = search[search.a == a]
    ax.plot(part.modulus, part.entropy, marker=".")
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set(title=f"a={a}", xlabel="modulus m", ylabel="conditional entropy (bits)")
fig.suptitle("Uncertainty remaining after p mod m is known")
fig.tight_layout()
plt.show()

> Which moduli contain all the information? Can you spot multiples of a smaller, fundamental modulus in the plots?

In [ ]:
discovered = {
    a: find_smallest_predictive_modulus(a, primes, MAX_MODULUS)
    for a in A_VALUES
}
discovered_table = pd.DataFrame({
    "a": A_VALUES,
    "smallest discovered modulus": [discovered[a] for a in A_VALUES],
})
discovered_table

### Empirical lookup rules

The four panels below are obtained from the discovered moduli—not from formulas supplied in advance. A bar at 0 or 1 says that the residue class determines the sign.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
lookup_tables = {}
for ax, a in zip(axes.flat, A_VALUES):
    m = discovered[a]
    table = conditional_legendre_table(a, primes, m)
    lookup_tables[a] = table
    ax.bar(table.residue.astype(str), table.fraction_plus)
    ax.set(title=f"a={a}, discovered m={m}", xlabel=f"p mod {m}",
           ylabel="P((a/p)=1)", ylim=(-0.05, 1.05))
fig.suptitle("Conditional probabilities at the discovered moduli")
fig.tight_layout()
plt.show()

for a in A_VALUES:
    m = discovered[a]
    rule = lookup_tables[a][["residue", "fraction_plus"]].copy()
    rule["empirical symbol"] = np.where(rule.fraction_plus > .5, "+1", "-1")
    print(f"a={a}, modulus {m}")
    display(rule[["residue", "empirical symbol"]].set_index("residue").T)

### What the data suggests

The lookup for $a=2$ gives

$$\left(\frac2p\right)=1\quad\Longleftrightarrow\quad p\equiv1,7\pmod8.$$

Only now do we recognize the supplementary law

$$\left(\frac2p\right)=(-1)^{(p^2-1)/8}.$$

For $a=3$, the data gives $+1$ in classes $1,11\pmod{12}$ and $-1$ in classes $5,7\pmod{12}$.

For $a=5$, it gives $+1$ in classes $1,4\pmod5$ and $-1$ in classes $2,3\pmod5$.

> Why should “5 is a square modulo $p$” be equivalent to “$p$ is a square modulo 5”? Do not answer from memory; compare the two columns below.

In [ ]:
five = conditional_legendre_table(5, primes, 5)[["residue", "fraction_plus"]]
five["empirical (5/p)"] = np.where(five.fraction_plus > .5, 1, -1)
five["(p/5)"] = five.residue.map(lambda r: legendre_symbol(r, 5))
five

### Why $a=7$ needs more than modulo 7

If modulo 7 were enough, every row below would have conditional probability 0 or 1. Compare it with the deterministic modulo-28 lookup already found.

> Why does information modulo 4 matter when $a$ itself is 7?

In [ ]:
display(conditional_legendre_table(7, primes, 7))
display(conditional_legendre_table(7, primes, discovered[7]))

## 8. Experiment 10 — compare two primes directly

For distinct odd primes $p,q$, compute both $(p/q)$ and $(q/p)$ independently with Euler's criterion. We then group their product by the two residues modulo 4.

In [ ]:
pair_primes = primes[(primes > 2)][:500]
pair_records = []
for i, p in enumerate(pair_primes):
    for q in pair_primes[i + 1:]:
        product = legendre_symbol(p, q) * legendre_symbol(q, p)
        pair_records.append((int(p % 4), int(q % 4), product))

pair_data = pd.DataFrame(pair_records, columns=["p mod 4", "q mod 4", "product"])
reciprocity_summary = pair_data.groupby(["p mod 4", "q mod 4"])["product"].agg(
    pairs="size", mean="mean", unique_values=lambda s: sorted(s.unique())
)
reciprocity_summary

> What changes when both primes are $3\bmod4$? Could this table have led us to the theorem?

The data suggests, and the law of **quadratic reciprocity** states,

$$\boxed{\left(\frac pq\right)\left(\frac qp\right)
=(-1)^{\frac{p-1}{2}\frac{q-1}{2}}}.$$

Equivalently, the two symbols agree unless both primes are $3\pmod4$, when their signs are opposite.

## 9. Experiments 11–12 — the Legendre-symbol matrix

Let $M_{ij}=(p_i/p_j)$, with a zero diagonal. In natural prime order it should look noisy. Reordering by residue modulo 4 exposes the symmetric and antisymmetric blocks predicted by the newly discovered law.

In [ ]:
def legendre_matrix(prime_array):
    ps = np.asarray(prime_array, dtype=np.int64)
    M = np.zeros((len(ps), len(ps)), dtype=np.int8)
    for i, p in enumerate(ps):
        for j, q in enumerate(ps):
            if i != j:
                M[i, j] = legendre_symbol(p, q)
    return M

matrix_primes = primes[(primes > 2)][:250]
M = legendre_matrix(matrix_primes)

fig, ax = plt.subplots(figsize=(7, 7))
image = ax.imshow(M, cmap="coolwarm", vmin=-1, vmax=1, interpolation="nearest")
ax.set(title=r"Natural order: $M_{ij}=(p_i/p_j)$", xlabel="j", ylabel="i")
fig.colorbar(image, ax=ax, ticks=[-1, 0, 1])
plt.show()

In [ ]:
order = np.argsort(matrix_primes % 4)  # class 1 first, then class 3
reordered_primes = matrix_primes[order]
M4 = M[np.ix_(order, order)]
split = int(np.sum(reordered_primes % 4 == 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, matrix, title in zip(axes, [M4, M4 - M4.T],
                             ["Reordered M", r"Difference $M-M^T$"]):
    im = ax.imshow(matrix, cmap="coolwarm", interpolation="nearest",
                   vmin=-2 if "Difference" in title else -1,
                   vmax=2 if "Difference" in title else 1)
    ax.axhline(split - .5, color="black", linewidth=1)
    ax.axvline(split - .5, color="black", linewidth=1)
    ax.set(title=title, xlabel="j", ylabel="i")
    fig.colorbar(im, ax=ax)
fig.suptitle("Primes 1 mod 4 first, primes 3 mod 4 second")
fig.tight_layout()
plt.show()

## 10. Experiment 13 — train/test prediction

An arithmetic rule should generalize to unseen primes. We learn a majority sign per training residue class and test it on later primes.

In [ ]:
def fit_residue_predictor(a, train_primes, m):
    table = conditional_legendre_table(a, train_primes, m)
    return {int(row.residue): (1 if row.plus_count >= row.minus_count else -1)
            for row in table.itertuples()}

def predict_residue_symbol(model, p, m):
    return model.get(int(p) % m)

def evaluate_residue_predictor(a, train_primes, test_primes, m):
    model = fit_residue_predictor(a, train_primes, m)
    ps, actual = valid_prime_symbol_arrays(a, test_primes)
    usable = [(p, x) for p, x in zip(ps, actual)
              if gcd(int(p), a * m) == 1 and predict_residue_symbol(model, p, m) is not None]
    predicted = np.array([predict_residue_symbol(model, p, m) for p, _ in usable])
    truth = np.array([x for _, x in usable])
    return len(usable), float(np.mean(predicted == truth))

cut = len(primes) // 2
train_primes, test_primes = primes[:cut], primes[cut:]
generalization = []
for m in [4, 7, 14, 28]:
    tested, accuracy = evaluate_residue_predictor(7, train_primes, test_primes, m)
    generalization.append((m, tested, accuracy))
pd.DataFrame(generalization, columns=["modulus", "test observations", "test accuracy"])

## 11. Experiment 14 — extend beyond four values

We repeat the same blind search for squarefree $a$. Some conductors exceed 60, so this extension searches through 120. The result is compared with $a$, $4a$, and $8a$ only after discovery.

In [ ]:
A_EXTENDED = [2, 3, 5, 6, 7, 10, 11, 13, 14, 15, 17, 19, 21, 23]
EXTENDED_MAX_MODULUS = 120

extended_rows = []
for a in A_EXTENDED:
    if a not in symbol_cache:
        symbol_cache[a] = valid_prime_symbol_arrays(a)
    m = find_smallest_predictive_modulus(a, primes, EXTENDED_MAX_MODULUS)
    extended_rows.append({
        "a": a,
        "discovered modulus": m,
        "entropy": conditional_entropy(a, primes, m) if m else np.nan,
        "accuracy": prediction_accuracy(a, primes, m) if m else np.nan,
        "a": a,
        "4a": 4 * a,
        "8a": 8 * a,
    })
extended_summary = pd.DataFrame(extended_rows)
extended_summary.style.format({"entropy": "{:.3g}", "accuracy": "{:.3f}"})

> Is there a systematic rule governing the discovered modulus? Why is it sometimes $a$, sometimes $4a$, and sometimes smaller than either?

The advanced interpretation is that, after removing square factors, the map

$$p\longmapsto\left(\frac ap\right)$$

is a **quadratic Dirichlet character** away from primes dividing its modulus. Its fundamental modulus is tied to the discriminant of the quadratic field $\mathbb{Q}(\sqrt a)$. Our entropy search has recovered that periodic character from samples, without using reciprocity to label them.

## 12. Information gain and simultaneous characters

For a binary variable, $I(X_a;R_m)=H(X_a)-H(X_a\mid R_m)$. We also combine the four signs into a vector. Their common period divides

$$\operatorname{lcm}(8,12,5,28)=840.$$

In [ ]:
def entropy_of_symbols(xs):
    q = float(np.mean(np.asarray(xs) == 1))
    return binary_entropy(q)

fig, ax = plt.subplots()
for a in A_VALUES:
    base_entropy = entropy_of_symbols(symbol_cache[a][1])
    part = search[search.a == a]
    ax.plot(part.modulus, base_entropy - part.entropy, label=f"a={a}")
ax.set(xlabel="modulus m", ylabel="information gain (bits)",
       title="How much does p mod m tell us?")
ax.legend()
plt.show()

vector_primes = np.array([p for p in primes if p > 7], dtype=np.int64)
vectors = [tuple(legendre_symbol(a, p) for a in A_VALUES) for p in vector_primes]
counts = Counter(vectors)
vector_table = pd.DataFrame([
    {"vector": str(v), "count": c, "frequency": c / len(vectors), "uniform benchmark": 1/16}
    for v, c in sorted(counts.items())
]).sort_values("vector")
display(vector_table)

pd.DataFrame({"p": vector_primes[:32], "p mod 840": vector_primes[:32] % 840,
              "sign vector": vectors[:32]})

## Final takeaway

$$\text{apparently random signs}
\quad\downarrow\quad
\text{statistical conditioning}
\quad\downarrow\quad
\text{perfect congruence laws}
\quad\downarrow\quad
\text{quadratic reciprocity}.$$

Quadratic residuosity looks random when primes are viewed without additional information. Once the correct congruence information is revealed, the randomness becomes deterministic arithmetic structure.

> Which part of the law could you have conjectured from global frequencies alone? Which experiments supplied genuinely new information?